# FoodScholar — Building the Graph

A runnable, release-ready tour of the `foodscholar` library against the real local corpus. **Every section is self-contained** — each one starts with a `bootstrap()` call that builds whatever state it needs, so you can run any section on its own.

- **§1 Quickstart** — five-line happy path against the in-memory backend.
- **§2 Config in code** — build a `FoodScholar` from a Python dict (no sister YAML).
- **§3 Stores: init Elastic + Neo4j** — the production-grade `fs.init()` pipeline.
- **§4 Ontology** — load real FoodOn, browse labels, ancestors, descendants.
- **§5 Corpus** — read the real CSV corpus under `data/foodscholar/corpus/`.
- **§6 Annotate** — GLiNER + HNSW (BioLORD) end-to-end, including the parquet snapshot.
- **§7 Push to Elastic** — upsert annotated chunks into the live ES index and round-trip a search.
- **§8 Inspect the graph** — `fs.graph` walk + neighbor lookups.

Run from the repo root:  `conda activate foodscholar && jupyter notebook notebooks/build_graph.ipynb`


## Setup

Run this once. It makes the package importable and defines `bootstrap()` — the helper every section below uses to get a ready `FoodScholar`.

**For the full pipeline**:

```
pip install -e '.[annotate,elastic,neo4j,ontology]'
# local services: Elasticsearch on http://localhost:9200, Neo4j on bolt://localhost:7687 (password: change-me)
```

Without those, in-memory / fixture paths still work — each cell logs which path it took.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import foodscholar
from foodscholar.logging import configure_logging
configure_logging(level="INFO")
print("foodscholar", foodscholar.__version__)

In [ ]:
from foodscholar import FoodScholar, FoodOnAPI
from foodscholar.io.chunk import Chunk
from foodscholar.ontology import load_ontology

# Real FoodOn is preferred. Drop the release at data/foodon.owl; pronto parses
# it once and caches at data/foodon_cache.parquet. Falls back to the tiny test
# fixture so the notebook still opens.
REAL_FOODON = REPO_ROOT / "data" / "foodon.owl"
MINI_FOODON = REPO_ROOT / "tests" / "fixtures" / "mini_foodon.obo"
USING_REAL_FOODON = REAL_FOODON.exists()

# The local CSV corpus (chunks_*.csv) lives under data/foodscholar/corpus/.
# That folder is a symlink to /mnt/workspaces/foodscholar/corpus on this host.
LOCAL_CORPUS_DIR = REPO_ROOT / "data" / "foodscholar" / "corpus"
HAS_LOCAL_CORPUS = LOCAL_CORPUS_DIR.exists() and any(LOCAL_CORPUS_DIR.glob("*.csv"))


# Tiny synthetic corpus — used when the real one is not on disk.
SAMPLE_CHUNKS = [
    Chunk(chunk_id="c1", text="Mediterranean diet rich in olive oil reduces cardiovascular risk.",
          source_doc_id="d1", source_type="abstract", section_type="abstract", year=2024),
    Chunk(chunk_id="c2", text="Whole grain consumption is associated with lower mortality.",
          source_doc_id="d2", source_type="abstract", section_type="results", year=2023),
    Chunk(chunk_id="c3", text="Peanut allergy management guidelines for paediatric populations.",
          source_doc_id="d3", source_type="guide",    section_type="guideline", year=2022),
    Chunk(chunk_id="c4", text="Quinoa improves glycemic control in adults with type 2 diabetes.",
          source_doc_id="d4", source_type="abstract", section_type="results", year=2020),
    Chunk(chunk_id="c5", text="Extra virgin olive oil contains polyphenols that reduce inflammation markers.",
          source_doc_id="d5", source_type="abstract", section_type="discussion", year=2019),
    Chunk(chunk_id="c6", text="School-based peanut allergy education reduces accidental exposures among children.",
          source_doc_id="d6", source_type="guide",    section_type="guideline", year=2021),
]


def _load_food_ontology() -> FoodOnAPI:
    if USING_REAL_FOODON:
        terms = load_ontology(REAL_FOODON,
                              cache_path=REPO_ROOT / "data" / "foodon_cache.parquet")
        return FoodOnAPI(terms, prefix_filter=["FOODON:"])
    return FoodOnAPI(load_ontology(MINI_FOODON), prefix_filter=None)


def _in_memory_config(*, snapshot: Path | None = None) -> dict:
    """The shape used by FoodScholar.from_config({...}) — no YAML needed."""
    return {
        "corpus": {
            "chunks_path": str(LOCAL_CORPUS_DIR) if HAS_LOCAL_CORPUS else "data/chunks.parquet",
            **({"annotated_snapshot_path": str(snapshot)} if snapshot else {}),
        },
        "ontology": {
            "foodon_path": str(REAL_FOODON if USING_REAL_FOODON else MINI_FOODON),
            "cache_path": str(REPO_ROOT / "data" / "foodon_cache.parquet"),
            "prefix_filter": ["FOODON:"] if USING_REAL_FOODON else None,
        },
        "annotate": {
            "ner": "gliner",
            "batch_size": 16,
            "linker": {
                "nel_backend": "hnsw",
                "nel_encoder": "biolord",
                "nel_min_sim": 0.70,
            },
        },
        "storage": {
            "chunk_store": {"backend": "memory"},
            "graph_store": {"backend": "memory"},
        },
    }


def bootstrap(*, with_chunks: bool = False, with_ontology: bool = True) -> FoodScholar:
    """Build a ready in-memory FoodScholar. Every section calls this so it
    can run standalone. with_ontology attaches FoodOn (real if available);
    with_chunks loads the sample corpus."""
    fs = FoodScholar.in_memory()
    if with_ontology:
        fs.attach_ontology(_load_food_ontology())
    if with_chunks:
        fs.upsert_chunks(SAMPLE_CHUNKS)
    return fs


print("FoodOn:",
      "REAL (data/foodon.owl)" if USING_REAL_FOODON
      else "mini test fixture — drop the real release at data/foodon.owl")
print(f"Local CSV corpus: {'YES' if HAS_LOCAL_CORPUS else 'no'}  ({LOCAL_CORPUS_DIR})")
print("bootstrap() ready")

## 1. Quickstart

`FoodScholar.in_memory()` needs no config and no services — the smallest happy path.


In [ ]:
fs = FoodScholar.in_memory()
fs.upsert_chunks([
    Chunk(chunk_id="c1", text="Mediterranean diet reduces cardiovascular risk.",
          source_doc_id="d1", source_type="abstract", section_type="abstract"),
])
fs.info()

## 2. Config in code — no YAML file required

The library accepts three equivalent config shapes: a Python dict, a `FoodScholarConfig` instance, or a YAML path. Below we build the facade directly from a dict — handy for notebooks and tests that don't want a sister YAML file.

`fs.info()` reflects what was wired.

In [ ]:
fs = FoodScholar.from_config({
    "corpus": {"chunks_path": "data/chunks.parquet"},
    "annotate": {"batch_size": 8},
    "storage": {
        "chunk_store": {"backend": "memory"},
        "graph_store": {"backend": "memory"},
    },
})
print(fs.info())
print("batch_size:", fs.config.annotate.batch_size)

## 3. Stores: init Elastic + Neo4j

Production stack: Elasticsearch 8.x for chunks, Neo4j 5.x for the graph. `fs.init()` is the one-call pipeline — it creates the ES index with the correct mappings and the Neo4j unique constraints on `(:Shelf), (:Theme), (:Card), (:Chunk)`. Both calls are idempotent, so re-running this cell is safe.

Credentials read from config (`storage.chunk_store.{api_key|username|password}`, `storage.graph_store.{user,password}`) with env fallback to `ELASTICSEARCH_API_KEY` and `NEO4J_PASSWORD` when unset.

This cell tries to connect to the local services. If they aren't running the cell prints the error and skips — the rest of the notebook keeps working against in-memory stores.

In [ ]:
import os
from foodscholar import FoodScholar

# Inline config — adjust to your local setup.
PROD_CONFIG = {
    "corpus": {"chunks_path": "data/chunks.parquet"},
    "ontology": {
        "foodon_path": str(REAL_FOODON if USING_REAL_FOODON else MINI_FOODON),
        "cache_path": str(REPO_ROOT / "data" / "foodon_cache.parquet"),
        "prefix_filter": ["FOODON:"] if USING_REAL_FOODON else None,
    },
    "annotate": {"ner": "gliner", "linker": {"nel_backend": "hnsw", "nel_encoder": "biolord"}},
    "storage": {
        "chunk_store": {
            "backend": "elastic",
            "url": "http://localhost:9200",
            "index": "foodscholar_chunks",
            # Add api_key / username+password here when your cluster needs them.
        },
        "graph_store": {
            "backend": "neo4j",
            "url": "bolt://localhost:7687",
            "user": "neo4j",
            "password": os.environ.get("NEO4J_PASSWORD", "change-me"),
        },
    },
}

try:
    fs_prod = FoodScholar.from_config(PROD_CONFIG)
    fs_prod.init()
    print("init OK:", fs_prod.info())
    PROD_OK = True
except Exception as e:
    print("init skipped —", type(e).__name__, str(e)[:200])
    print("Start local services first:")
    print("  docker run -d -p 9200:9200 -e discovery.type=single-node \\")
    print("      -e xpack.security.enabled=false elasticsearch:8.13.0")
    print("  docker run -d -p 7687:7687 -p 7474:7474 \\")
    print("      -e NEO4J_AUTH=neo4j/change-me neo4j:5")
    PROD_OK = False
    fs_prod = None

## 4. Ontology

`fs.ontology` is the FoodOn lookup API — labels, synonyms, ancestors, descendants. It backs the linker (HNSW term index) and the Layer A backbone. Uses the real FoodOn when `data/foodon.owl` is present, otherwise the small test fixture.

*Self-contained: just run the cell.*

In [ ]:
fs = bootstrap()
ont = fs.ontology
print(f"{len(ont)} terms loaded")

olive_oil = ont.name_to_id("olive oil")
print("olive oil ->", olive_oil)
if olive_oil:
    print("ancestors ->", [ont.id_to_label(a) for a in ont.id_to_ancestors(olive_oil)])

fruit = ont.name_to_id("fruit")
if fruit:
    descs = ont.id_to_descendants(fruit)
    print(f"fruit has {len(descs)} descendants, e.g.",
          [ont.id_to_label(d) for d in descs[:8]])

## 5. Corpus

The library reads CSV (legacy corpus), Parquet, or JSONL — all from the same `fs.load_chunks(path)` / `iter_chunks(path)` entry point. CSV chunks follow `(chunk_id, chunk_text, type, chunk_metadata)`. Field-size limit is raised to 10 MB to handle the largest abstracts.

Below: if the local corpus exists (under `data/foodscholar/corpus/`), we read it and report a per-file count. Otherwise we fall back to `SAMPLE_CHUNKS`.

*Self-contained: just run the cell.*

In [ ]:
from collections import Counter
from foodscholar.corpus import iter_chunks

fs = bootstrap()

if HAS_LOCAL_CORPUS:
    counts = Counter()
    total = 0
    files = sorted(LOCAL_CORPUS_DIR.glob("*.csv"))
    print(f"reading {len(files)} CSV files from {LOCAL_CORPUS_DIR}")
    for path in files:
        n = sum(1 for _ in iter_chunks(path))
        counts[path.name] = n
        total += n
    print(f"\ntotal chunks across all files: {total}\n")
    for fname, n in counts.most_common(8):
        print(f"  {n:5d}  {fname}")
    if len(counts) > 8:
        print(f"  ...  ({len(counts) - 8} more files)")
else:
    fs.upsert_chunks(SAMPLE_CHUNKS)
    for c in fs.chunk_store.scan():
        print(f"  {c.chunk_id} [{c.source_type}/{c.section_type}] {c.text[:55]}...")

## 6. Annotate — GLiNER + HNSW end-to-end

`fs.load_and_annotate(path)` is the release-ready entry point: it loads chunks, runs GLiNER bio in batches of 16, encodes mention surface forms with BioLORD, looks them up via the FoodOn HNSW index (built on first use, cached at `data/foodon_hnsw_<encoder>_<sig>.bin`), and writes the enriched chunks back. If `cfg.corpus.annotated_snapshot_path` is set, a parquet snapshot lands beside the chunk store; subsequent runs short-circuit if the snapshot already exists.

This cell runs the real GLiNER + BioLORD pipeline when the `[annotate]` extra is installed; otherwise it explains what is missing.

In [ ]:
import importlib.util as _u
from pathlib import Path

has_gliner = bool(_u.find_spec("gliner"))
has_hnswlib = bool(_u.find_spec("hnswlib"))
has_sbert = bool(_u.find_spec("sentence_transformers"))

if not (has_gliner and has_hnswlib and has_sbert):
    print("Annotate skipped — needs:  pip install -e '.[annotate]'")
    print(f"  gliner installed: {has_gliner}")
    print(f"  hnswlib installed: {has_hnswlib}")
    print(f"  sentence-transformers installed: {has_sbert}")
else:
    # Pick one local file so first-run isn't all the corpus at once.
    if HAS_LOCAL_CORPUS:
        chunks_path = next(iter(sorted(LOCAL_CORPUS_DIR.glob("*.csv"))))
        print("annotating:", chunks_path.name)
    else:
        chunks_path = REPO_ROOT / "tests" / "fixtures" / "sample_chunks.jsonl"
        print("annotating:", chunks_path)

    snapshot = REPO_ROOT / "data" / "annotated_demo.parquet"
    snapshot.unlink(missing_ok=True)  # rerun the demo, don't short-circuit

    fs = FoodScholar.from_config(_in_memory_config(snapshot=snapshot))
    meta = fs.load_and_annotate(chunks_path)
    if meta is None:
        print("snapshot already exists — re-run skipped")
    else:
        print(f"annotated {meta.record_count} chunks  (artifact={meta.artifact_id})")

    # Show 3 sample annotated chunks.
    for c in fs.chunk_store.scan()[:3]:
        print(f"\n  {c.chunk_id} [{c.source_type}]  {c.text[:60]}...")
        for m in c.mentions[:5]:
            print(f"    mention: {m.text!r}  type={m.entity_type}  score={m.score:.2f}")
        for ln in c.entity_links[:5]:
            print(f"    link:    {ln.mention.text!r} -> {ln.ontology_id}  conf={ln.confidence:.2f}")
    print(f"\nsnapshot written to: {snapshot}  (exists={snapshot.exists()})")

## 7. Push annotated chunks to Elastic

Once chunks are annotated locally, the same `chunk_store.upsert()` call talks to Elasticsearch — the protocol is identical regardless of backend. This cell:

1. Re-uses the in-memory annotated store from §6 (or builds a small one from `SAMPLE_CHUNKS`).
2. Switches to an ES-backed `FoodScholar` from a dict config, runs `fs.init()` (idempotent — creates the index if missing).
3. Upserts the chunks.
4. Round-trips a search.

Requires a running Elasticsearch at `http://localhost:9200`.

In [ ]:
import importlib.util as _u

if not _u.find_spec("elasticsearch"):
    print("ES round-trip skipped — needs:  pip install -e '.[elastic]'")
elif not PROD_OK:
    print("ES round-trip skipped — §3 could not connect to local Elasticsearch.")
else:
    # Build a small annotated set in-memory (independent of §6 having run for real).
    fs_mem = bootstrap(with_chunks=True)
    try:
        fs_mem.annotate()
    except Exception as e:
        # GLiNER unavailable — push raw chunks anyway to exercise the ES path.
        print(f"annotate skipped in-memory ({type(e).__name__}); pushing raw chunks")

    fs_es = FoodScholar.from_config({
        **PROD_CONFIG,
        "storage": {
            "chunk_store": PROD_CONFIG["storage"]["chunk_store"],
            "graph_store": {"backend": "memory"},  # graph not needed for this round-trip
        },
    })
    fs_es.init()
    fs_es.upsert_chunks(list(fs_mem.chunk_store.scan()))
    print(f"upserted {len(fs_mem.chunk_store.scan())} chunks to ES index "
          f"'{fs_es.config.storage.chunk_store.index}'")

    hits = fs_es.chunk_store.search("olive oil", k=3, use_vector=False, use_bm25=True)
    print("\nBM25 search 'olive oil' top 3:")
    for h in hits:
        print(f"  {h.chunk_id}  {h.text[:80]}")

## 8. Inspect the graph

`fs.graph` is the read/write surface over the chunk + graph stores. Even when Layer A/B/C builders are stubs, you can build a small graph by hand to exercise the API.

In [ ]:
fs = bootstrap(with_chunks=True)

# Layer A — hand-build a couple of shelves.
fs.graph.add_shelf(shelf_id="s-med", label="Mediterranean diet",
                   facet="dietary_patterns", depth=0)
fs.graph.add_shelf(shelf_id="s-allergy", label="Food allergies",
                   facet="allergies", depth=0)
fs.graph.attach_chunks(["c1", "c5"], shelf="s-med")
fs.graph.attach_chunks(["c3", "c6"], shelf="s-allergy")

# Layer B — one theme.
fs.graph.add_theme(theme_id="t-olive", label="Olive oil benefits",
                   shelf_ids=["s-med"], discovered_by="leiden", discovery_version="nb")
fs.graph.attach_chunks(["c1", "c5"], theme="t-olive")

shelf = fs.graph.shelf("s-med")
print("shelf :", shelf.label, "| facet:", shelf.facet)
print("chunks:", [c.chunk_id for c in shelf.chunks()])
print("themes:", [t.label for t in shelf.themes()])

from foodscholar import make_artifact_meta
meta = make_artifact_meta(phase="notebook-build", config=fs.config, record_count=len(SAMPLE_CHUNKS))
print(f"\nartifact: {meta.artifact_id}  config_hash={meta.config_hash}")

## Status & next

**Implemented end-to-end:** in-code config (dict / `FoodScholarConfig` / YAML), Elastic + Neo4j adapters with full `init()` provisioning, GLiNER bio NER (batched), HNSW NEL over BioLORD-embedded FoodOn, parquet snapshotting, idempotent re-runs.

**Still stub:** Layer A / B / C builders and the retrieval pipeline — `fs.graph` is the API those will write through.

See `BRIEF.md` §2 for the locked architecture and §12 for the remaining milestones.